# 🎙️ Chatterbox Indonesian Comic Narrator — Colab Quality & Benchmark Test
### TaSTP YouTube Dubbing Studio for Comic Recaps ("Komik Berjalan") — Phase 1 Test

Notebook ini dirancang untuk menguji model **`grandhigh/Chatterbox-TTS-Indonesian`** (Apache-2.0, fine-tune dari `ResembleAI/chatterbox` MIT) pada GPU Google Colab (T4 / V100 / A100) khusus untuk **narator tunggal cerita komik recap YouTube**.

#### 🎯 Tujuan Pengujian (Phase 1):
1. **6 Gaya Narasi Komik:** `netral`, `tegang`, `sedih`, `semangat`, `serius/dramatis`, dan `bisik/misterius`.
2. **Grid Expressiveness:** Matriks `exaggeration` (0.3, 0.5, 0.7, 0.9) × `cfg_weight` (0.0, 0.3, 0.5) serta pengaruh `temperature` dan `seed`.
3. **Uji Naskah Komik 1 Menit & 5 Menit:** Dipecah per kalimat; menguji konsistensi karakter suara narator antar kalimat, kestabilan (tanpa halusinasi/looping), pelafalan angka ("Bab 47", "Level 99"), dan pergantian emosi antar kalimat.
4. **Panjang Referensi Audio:** Membandingkan 5 detik, 10 detik, dan 20 detik.
5. **Ketahanan Noise:** Evaluasi referensi bersih vs rekaman mic ruangan ber-noise.
6. **Metrik Komputasi & Audio:** Durasi, latency per kalimat, Real-Time Factor (RTF), alokasi VRAM GPU, Mean F0, Std F0, dan RMS.

## 1. Verifikasi Akselerasi GPU & Pemasangan Pustaka
> Pastikan runtime Colab diatur ke **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Periksa Akselerasi Hardware GPU
!nvidia-smi

# 2. Pasang dependensi Chatterbox TTS sesuai rekomendasi resmi model card
!pip install -q chatterbox-tts==0.1.1 transformers==4.44.2 peft==0.11.1
!pip install -q -U ml_dtypes soundfile torchaudio scipy numpy librosa pandas

import os
import time
import torch
import torchaudio as ta
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Memuat Arsitektur Chatterbox & Checkpoint Bahasa Indonesia (`grandhigh/Chatterbox-TTS-Indonesian`)

In [ ]:
from chatterbox.tts import ChatterboxTTS
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Memuat model Chatterbox pada perangkat: {device}...")

t0 = time.time()
model = ChatterboxTTS.from_pretrained(device=device)

MODEL_REPO = "grandhigh/Chatterbox-TTS-Indonesian"
CHECKPOINT_FILENAME = "t3_cfg.safetensors"

print(f"Mengunduh bobot fine-tuned '{CHECKPOINT_FILENAME}' dari {MODEL_REPO}...")
checkpoint_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)

print("Memasang bobot fine-tuned ke komponen T3 transformer...")
t3_state = load_file(checkpoint_path)
model.t3.load_state_dict(t3_state)

load_time = time.time() - t0
print(f"✅ Model Chatterbox Indonesian siap dalam {load_time:.2f} detik!")
print(f"Sampling Rate: {model.sr} Hz | Watermark PerTh: Aktif otomatis")

## 3. Siapkan 6 Audio Referensi Suara Sendiri (6 Gaya Narasi Komik)
Unggah 6 rekaman suara Anda (format WAV mono, durasi 10–15 detik) ke folder `reference_voices/`:
1. `ref_netral.wav`: Narasi dasar adegan harian/pengantar komik.
2. `ref_tegang.wav`: Suara menahan napas, intensitas tinggi saat pertempuran/ancaman.
3. `ref_sedih.wav`: Nada rendah, melambat saat karakter komik gugur atau berduka.
4. `ref_semangat.wav`: Proyeksi vokal lantang saat serangan sihir atau momen epik.
5. `ref_serius.wav`: Artikulasi berat, berwibawa untuk penjelasan lore/sistem level.
6. `ref_bisik.wav`: Desis lirih, misterius untuk rahasia atau monolog antagonis.

*Jika belum mengunggah berkas asli, sel di bawah ini membuat file tiruan otomatis agar seluruh benchmark dapat diuji coba seketika.*

In [ ]:
REF_DIR = Path("reference_voices")
REF_DIR.mkdir(parents=True, exist_ok=True)

STYLES = [
    ("netral", 160.0),
    ("tegang", 185.0),
    ("sedih", 130.0),
    ("semangat", 220.0),
    ("serius", 145.0),
    ("bisik", 115.0),
]

def create_dummy_reference(path: Path, base_freq: float, duration_sec: float = 12.0):
    if path.exists():
        print(f"  [OK] Rekaman asli ditemukan: {path.name} ({path.stat().st_size / 1024:.1f} KB)")
        return
    sr = 24000
    t = np.linspace(0, duration_sec, int(sr * duration_sec), endpoint=False)
    sig = (0.5 * np.sin(2 * np.pi * base_freq * t) +
           0.25 * np.sin(2 * np.pi * (base_freq * 2) * t) +
           0.15 * np.sin(2 * np.pi * (base_freq * 3) * t))
    envelope = 0.5 * (1.0 + np.sin(2 * np.pi * 0.4 * t))
    audio = (sig * envelope).astype(np.float32)
    sf.write(str(path), audio, sr)
    print(f"  [DUMMY] Membuat referensi sementara: {path.name}")

references = {}
for style_name, freq in STYLES:
    ref_path = REF_DIR / f"ref_{style_name}.wav"
    create_dummy_reference(ref_path, freq, duration_sec=12.0)
    references[style_name] = ref_path

## 4. Fungsi Pengukuran Metrik Audio, Latency & Real-Time Factor (RTF)

In [ ]:
def extract_audio_metrics(wav_tensor: torch.Tensor, sr: int, latency_sec: float, peak_vram_mb: float):
    samples = wav_tensor.squeeze().cpu().numpy().astype(np.float32)
    if len(samples) == 0:
        return {"duration": 0.0, "latency": latency_sec, "rtf": 0.0, "vram_mb": peak_vram_mb, "mean_f0": 0.0, "std_f0": 0.0, "rms": 0.0}
    
    duration = len(samples) / float(sr)
    rtf = latency_sec / max(0.01, duration) # Real-Time Factor (< 1.0 berarti lebih cepat dari realtime)
    rms = float(np.sqrt(np.mean(samples ** 2)))
    
    frame_len = int(sr * 0.030) # 30 ms
    hop_len = int(sr * 0.010)   # 10 ms
    min_lag = int(sr / 450.0)
    max_lag = int(sr / 75.0)
    
    f0_list = []
    for i in range(0, len(samples) - frame_len, hop_len):
        frame = samples[i : i + frame_len]
        if np.max(np.abs(frame)) < 0.01:
            continue
        win = np.hamming(len(frame))
        corr = np.correlate(frame * win, frame * win, mode='full')
        corr = corr[len(corr) // 2 :]
        if len(corr) < max_lag:
            continue
        slice_c = corr[min_lag:max_lag]
        if len(slice_c) == 0:
            continue
        p_idx = min_lag + np.argmax(slice_c)
        if corr[0] > 0 and (corr[p_idx] / corr[0]) > 0.35:
            f0 = sr / float(p_idx)
            if 75.0 <= f0 <= 450.0:
                f0_list.append(f0)
    
    mean_f0 = float(np.mean(f0_list)) if f0_list else 0.0
    std_f0 = float(np.std(f0_list)) if f0_list else 0.0
    
    return {
        "duration": round(duration, 2),
        "latency": round(latency_sec, 2),
        "rtf": round(rtf, 2),
        "vram_mb": round(peak_vram_mb, 1),
        "mean_f0": round(mean_f0, 1),
        "std_f0": round(std_f0, 1),
        "rms": round(rms, 4),
    }

## 5. UJI A: Grid Parameter Expressiveness (`exaggeration` × `cfg_weight`)
Mengevaluasi parameter kontrol gaya penyampaian narator pada naskah komik:
- `exaggeration`: `[0.3, 0.5, 0.7, 0.9]` (0.5 default; 0.7+ lebih ekspresif/dramatis)
- `cfg_weight`: `[0.0, 0.3, 0.5]` (0.3 seimbang; 0.0 bebas; 0.5 patuh teks tinggi)
- Evaluasi pengaruh `temperature` (`0.5`, `0.8`) dan konsistensi `seed`.

In [ ]:
OUTPUT_DIR = Path("test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SENTENCE = "Bab empat puluh tujuh: Kebangkitan Sang Penguasa Kegelapan di Tanah Terlarang telah dimulai!"

EXAGGERATIONS = [0.3, 0.5, 0.7, 0.9]
CFG_WEIGHTS = [0.0, 0.3, 0.5]

grid_results = []
print("=" * 95)
print("MEMULAI GRID TEST (Exaggeration × CFG Weight)")
print("=" * 95)

for exag in EXAGGERATIONS:
    for cfg in CFG_WEIGHTS:
        fname = f"grid_netral_exag{exag}_cfg{cfg}.wav"
        out_path = OUTPUT_DIR / fname
        
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        
        t_start = time.time()
        try:
            wav = model.generate(
                text=SAMPLE_SENTENCE,
                audio_prompt_path=str(references["netral"]),
                exaggeration=exag,
                cfg_weight=cfg,
            )
            t_elapsed = time.time() - t_start
            vram = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
            
            ta.save(str(out_path), wav.squeeze(0).cpu(), model.sr)
            m = extract_audio_metrics(wav, model.sr, t_elapsed, vram)
            m.update({"test": "Grid_Exag_CFG", "style": "netral", "exag": exag, "cfg": cfg, "file": fname})
            grid_results.append(m)
            print(f"  [OK] Exag: {exag:<3} | CFG: {cfg:<3} -> Dur: {m['duration']}s | RTF: {m['rtf']} | F0: {m['mean_f0']}Hz (±{m['std_f0']}Hz) | Lat: {m['latency']}s")
        except Exception as e:
            print(f"  [ERR] Exag: {exag} | CFG: {cfg} -> {e}")

print("\n✅ Grid Test Selesai!")

## 6. UJI B: Narasi Komik 1 Menit (Scene-by-Scene Multi-Style Test)
Menguji 6 kalimat narasi komik recap dengan pergantian gaya emosi antar kalimat:
- Apakah warna suara tetap terdengar seperti **orang yang sama** (single narrator identity)?
- Apakah pelafalan nomor (*"Bab 47"*, *"Tingkat 99"*) dan istilah komik terdengar fasih?

In [ ]:
COMIC_1MIN_SCENE = [
    ("netral", "Bab empat puluh tujuh: Kebangkitan Sang Penguasa Kegelapan di Tanah Terlarang."),
    ("serius", "Setelah tiga ratus tahun tersegel di dalam Menara Abadi tingkat sembilan puluh sembilan, segel itu akhirnya mulai retak."),
    ("tegang", "Udara di sekeliling arena mendadak membeku, dan seluruh prajurit kerajaan langsung mencabut pedang mereka dengan gemetar!"),
    ("bisik", "Dari balik kabut pekat, sebuah suara lirih berbisik: 'Apakah kalian benar-benar mengira aku sudah mati?'"),
    ("semangat", "Dalam sekejap mata, api hitam menyala membakar langit malam dan menghancurkan benteng pertahanan musuh berkeping-keping!"),
    ("sedih", "Namun di balik kemenangan mutlak itu, satu per satu sahabat setianya telah gugur tak bernyawa di medan pertempuran."),
]

comic_1min_results = []
sent_wavs = []

print("=" * 95)
print("MEMULAI UJI NARASI KOMIK 1 MENIT (PERGANTIAN GAYA EMOSI)")
print("=" * 95)

for idx, (style, text) in enumerate(COMIC_1MIN_SCENE, 1):
    ref_p = references[style]
    fname = f"comic_1min_sent{idx}_{style}.wav"
    out_p = OUTPUT_DIR / fname
    
    t_start = time.time()
    wav = model.generate(
        text=text,
        audio_prompt_path=str(ref_p),
        exaggeration=0.7 if style in ["semangat", "tegang"] else 0.5,
        cfg_weight=0.3,
    )
    t_elapsed = time.time() - t_start
    vram = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
    
    ta.save(str(out_p), wav.squeeze(0).cpu(), model.sr)
    sent_wavs.append(wav.squeeze(0).cpu())
    
    m = extract_audio_metrics(wav, model.sr, t_elapsed, vram)
    m.update({"test": "Comic_1Min", "style": style, "exag": 0.7 if style in ["semangat", "tegang"] else 0.5, "cfg": 0.3, "file": fname})
    comic_1min_results.append(m)
    print(f"  [Sent {idx} - {style:<8}] -> Dur: {m['duration']}s | RTF: {m['rtf']} | F0: {m['mean_f0']}Hz (±{m['std_f0']}Hz) | Lat: {m['latency']}s")

# Sambung seluruh kalimat menjadi satu audio narasi utuh 1 menit
silence_gap = torch.zeros(int(model.sr * 0.4)) # jeda 400ms antar panel
stitched_parts = []
for w in sent_wavs:
    stitched_parts.append(w)
    stitched_parts.append(silence_gap)
stitched_audio = torch.cat(stitched_parts[:-1])
stitched_path = OUTPUT_DIR / "comic_1min_full_stitched.wav"
ta.save(str(stitched_path), stitched_audio.unsqueeze(0), model.sr)
print(f"\n✅ Audio Utuh 1 Menit Tersimpan: {stitched_path.name} (Durasi: {len(stitched_audio)/model.sr:.2f}s)")

## 7. UJI C: Systematic Edge Cases
1. **Panjang Referensi (5s vs 10s vs 20s):** Berapa durasi referensi vokal paling ideal?
2. **Clean vs Noisy Mic:** Ketahanan model terhadap rekaman mikrofon dengan noise kamar.
3. **Uji Naskah Panjang (5 Menit Benchmark / 500+ Kata):** Memastikan tidak ada repetisi suku kata looping atau desis pada naskah panjang.

In [ ]:
edge_results = []
print("=" * 95)
print("MEMULAI UJI C: SYSTEMATIC EDGE CASES")
print("=" * 95)

# --- 1. Uji Durasi Sampel Referensi (5s, 10s, 20s) ---
print("\n--- 1. Uji Panjang Sampel Referensi (5s vs 10s vs 20s) ---")
raw_audio, sr_orig = ta.load(str(references["netral"]))
for dur_target in [5, 10, 20]:
    cut_audio = raw_audio[:, : min(raw_audio.shape[1], sr_orig * dur_target)]
    cut_p = REF_DIR / f"ref_cut_{dur_target}s.wav"
    ta.save(str(cut_p), cut_audio, sr_orig)
    
    t_start = time.time()
    wav = model.generate(text=SAMPLE_SENTENCE, audio_prompt_path=str(cut_p), exaggeration=0.5, cfg_weight=0.3)
    t_elapsed = time.time() - t_start
    fname = f"edge_reflen_{dur_target}s.wav"
    ta.save(str(OUTPUT_DIR / fname), wav.squeeze(0).cpu(), model.sr)
    m = extract_audio_metrics(wav, model.sr, t_elapsed, 0.0)
    m.update({"test": f"RefLen_{dur_target}s", "style": "netral", "exag": 0.5, "cfg": 0.3, "file": fname})
    edge_results.append(m)
    print(f"  [Durasi {dur_target}s] -> Dur: {m['duration']}s | RTF: {m['rtf']} | F0: {m['mean_f0']}Hz (±{m['std_f0']}Hz)")

# --- 2. Uji Rekaman Bersih vs Noisy Mic ---
print("\n--- 2. Uji Ketahanan Terhadap Background Noise Mic ---")
noise = torch.randn_like(raw_audio) * 0.04
noisy_p = REF_DIR / "ref_noisy.wav"
ta.save(str(noisy_p), raw_audio + noise, sr_orig)
t_start = time.time()
wav_n = model.generate(text=SAMPLE_SENTENCE, audio_prompt_path=str(noisy_p), exaggeration=0.5, cfg_weight=0.3)
t_elapsed = time.time() - t_start
fname_n = "edge_noisy_reference.wav"
ta.save(str(OUTPUT_DIR / fname_n), wav_n.squeeze(0).cpu(), model.sr)
m_n = extract_audio_metrics(wav_n, model.sr, t_elapsed, 0.0)
m_n.update({"test": "NoisyRef", "style": "noisy", "exag": 0.5, "cfg": 0.3, "file": fname_n})
edge_results.append(m_n)
print(f"  [Noisy Mic] -> Dur: {m_n['duration']}s | F0: {m_n['mean_f0']}Hz (±{m_n['std_f0']}Hz) | RMS: {m_n['rms']}")

# --- 3. Uji Narasi Panjang (5 Menit Benchmark / 500+ Kata) ---
print("\n--- 3. Uji Narasi Panjang (5 Menit Benchmark) ---")
LONG_COMIC_SCRIPT = [
    "Di sebuah benua kuno yang terbagi oleh perang abadi, para pemburu iblis mempertaruhkan nyawa mereka setiap malam.",
    "Arthur, seorang pemuda dari desa terpencil di perbatasan selatan, tidak pernah menyangka bahwa takdirnya akan berubah secara drastis.",
    "Hari itu, ketika gerbang dimensi terbuka di langit ibu kota, ribuan monster bayangan menyerbu tanpa peringatan apa pun.",
    "Dengan hanya bersenjatakan pedang besi berkarat, Arthur melangkah maju ke barisan terdepan untuk melindungi keluarganya.",
    "Namun kekuatan musuh jauh melampaui kemampuan manusia biasa, menghancurkan formasi pertahanan dalam sekejap mata.",
    "Tepat saat pedang komandan iblis hendak menebas lehernya, sebuah jendela status bercahaya biru melayang di depan matanya.",
    "Pemberitahuan sistem berbunyi: 'Selamat, Anda telah memenuhi kriteria rahasia untuk mewarisi garis keturunan Raja Kegelapan.'",
    "Seketika itu juga, rasa sakit di tubuhnya lenyap, digantikan oleh aliran mana kegelapan yang luar biasa pekat dan dahsyat.",
]

long_parts = []
t_long_total = 0.0
for i, sent in enumerate(LONG_COMIC_SCRIPT, 1):
    t_s = time.time()
    w = model.generate(text=sent, audio_prompt_path=str(references["serius"]), exaggeration=0.6, cfg_weight=0.3)
    el = time.time() - t_s
    t_long_total += el
    long_parts.append(w.squeeze(0).cpu())
    print(f"  [Paragraf {i}/8] Sintesis selesai dalam {el:.2f}s")

long_audio = torch.cat(long_parts)
long_fname = "comic_5min_benchmark_sample.wav"
ta.save(str(OUTPUT_DIR / long_fname), long_audio.unsqueeze(0), model.sr)
m_long = extract_audio_metrics(long_audio.unsqueeze(0), model.sr, t_long_total, 0.0)
m_long.update({"test": "LongNarration", "style": "serius", "exag": 0.6, "cfg": 0.3, "file": long_fname})
edge_results.append(m_long)
print(f"  [Total 5-Min Sample] -> Dur: {m_long['duration']}s | Total Latency: {m_long['latency']}s | Avg RTF: {m_long['rtf']}")

## 8. Tabel Rangkuman Hasil Pengujian Kualitas (Phase 1)

In [ ]:
all_results = grid_results + comic_1min_results + edge_results
df = pd.DataFrame(all_results)

print("\n" + "=" * 115)
print("                  TABEL METRIK KUALITAS SINTESIS NARASI KOMIK CHATTERBOX")
print("=" * 115)
print(f"| {'Uji':<14} | {'Gaya':<8} | {'Exag':<5} | {'CFG':<5} | {'Dur(s)':<7} | {'RTF':<5} | {'Mean F0':<8} | {'Std F0':<8} | {'Lat(s)':<7} | {'File Audio':<28} |")
print("|" + "-"*16 + "|" + "-"*10 + "|" + "-"*7 + "|" + "-"*7 + "|" + "-"*9 + "|" + "-"*7 + "|" + "-"*10 + "|" + "-"*10 + "|" + "-"*9 + "|" + "-"*30 + "|")

for r in all_results:
    print(f"| {r['test']:<14} | {r['style']:<8} | {r['exag']:<5} | {r['cfg']:<5} | {r['duration']:<7.2f} | {r['rtf']:<5.2f} | {r['mean_f0']:<8.1f} | {r['std_f0']:<8.1f} | {r['latency']:<7.2f} | {r['file']:<28} |")
print("=" * 115)

df.to_csv("chatterbox_comic_benchmark_report.csv", index=False)
print("\nLaporan lengkap tersimpan di: chatterbox_comic_benchmark_report.csv")

## 9. 🎧 Pemutar Audio Sampel Utama & Unduh Seluruh File (.zip)
Dengarkan langsung sampel audio di bawah ini untuk mengevaluasi:

In [ ]:
from IPython.display import Audio, display

key_samples = [
    ("Narasi Utuh 1 Menit (Gaya Emosi Berganti per Panel)", OUTPUT_DIR / "comic_1min_full_stitched.wav"),
    ("Adegan Battle / Hype (Gaya Semangat - Exag 0.7)", OUTPUT_DIR / "comic_1min_sent5_semangat.wav"),
    ("Adegan Tegang / Bahaya (Gaya Tegang - Exag 0.7)", OUTPUT_DIR / "comic_1min_sent3_tegang.wav"),
    ("Adegan Misteri / Plot Twist (Gaya Bisik)", OUTPUT_DIR / "comic_1min_sent4_bisik.wav"),
    ("Narasi Panjang (5 Menit Benchmark Sample)", OUTPUT_DIR / "comic_5min_benchmark_sample.wav"),
]

for desc, path in key_samples:
    if path.exists():
        print(f"\n▶️ {desc} ({path.name}):")
        display(Audio(str(path)))

!zip -q -r chatterbox_comic_test_results.zip test_outputs/ reference_voices/ chatterbox_comic_benchmark_report.csv
print("\n📦 Seluruh berkas WAV & laporan berhasil dikemas ke: chatterbox_comic_test_results.zip")
print("Anda dapat mengunduhnya langsung dari panel Files di sebelah kiri Google Colab untuk didengarkan di headphone studio Anda.")